# grok-009 · GSM8K 硬评测尺子 + 基线（学习向）

> **对应深入层 D（任务闭环）+ 路线 3（系统评测）**  
> 004 的合成题太容易（JSON/SQL 常顶满）。本课换成社区标准 **GSM8K**：答案可抽取、可 exact match。
>
> **你在学什么**
> 1. 如何下载/抽样 GSM8K，并 **固定 seed 子集** 当 eval  
> 2. 统一 greedy 解码 + 数字抽取  
> 3. 只跑 **base 模型基线**（本课不训练）  
>
> **默认基座**：`Qwen/Qwen2.5-1.5B-Instruct`（评测快）；可选 3B。  
> **产物**：`gsm8k_eval_v1/` + `gsm8k_baseline.json`


In [ ]:
# 【步骤】评测用单卡更稳：锁 GPU0，减少多卡 device_map 干扰
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("CUDA_VISIBLE_DEVICES", os.environ.get("CUDA_VISIBLE_DEVICES"))


In [ ]:
# 【步骤】环境
import os, re, json, time, random, platform
from pathlib import Path
import torch

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

OUT = Path("/kaggle/working")
EVAL = OUT / "gsm8k_eval_v1"
EVAL.mkdir(parents=True, exist_ok=True)

assert torch.cuda.is_available()
print("torch", torch.__version__, "device", torch.cuda.get_device_name(0))
DEVICE = torch.device("cuda:0")
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"  # 可改 3B，但更慢
N_EVAL = 100  # 学习向子集；全量 1319 可自行加大


In [ ]:
# 【步骤】拉取 GSM8K test，固定抽样，落盘冻结（禁止训练泄漏到此文件）
from datasets import load_dataset

ds = load_dataset("openai/gsm8k", "main", split="test")
print("gsm8k test size", len(ds))
idx = list(range(len(ds)))
random.Random(SEED).shuffle(idx)
idx = idx[:N_EVAL]

rows = []
for i in idx:
    ex = ds[int(i)]
    # gold 通常在 #### 后
    ans = ex["answer"]
    m = re.search(r"####\s*([-0-9.,]+)", ans)
    gold = m.group(1).replace(",", "") if m else ans.strip().split()[-1]
    rows.append({
        "id": f"gsm8k_{i}",
        "question": ex["question"],
        "gold": gold,
        "full_answer": ans,
    })

def write_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

write_jsonl(EVAL / "gsm8k_em.jsonl", rows)
manifest = {
    "seed": SEED,
    "source": "openai/gsm8k:main/test",
    "n": len(rows),
    "metric": "exact_match_on_final_number",
    "model_id_baseline": MODEL_ID,
    "notes": "冻结评测集：训练数据不得包含这些 id/题面",
}
(EVAL / "manifest.json").write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print(manifest)
print("sample Q:", rows[0]["question"][:120], "gold:", rows[0]["gold"])


In [ ]:
# 【步骤】加载 base 模型（评测向：fp16 整模；1.5B 可塞进单 T4）
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map={"": 0}, trust_remote_code=True,
)
model.eval()
print("loaded", MODEL_ID)


In [ ]:
# 【步骤】生成 + 抽最终数字 + exact match
def extract_final_number(text: str):
    if not text:
        return None
    # 优先 ####，否则取最后一个数字串
    m = re.search(r"####\s*([-0-9.,]+)", text)
    if m:
        return m.group(1).replace(",", "")
    nums = re.findall(r"-?\d+(?:\.\d+)?", text.replace(",", ""))
    return nums[-1] if nums else None

@torch.no_grad()
def generate(question: str, max_new=256) -> str:
    # 要求模型按 GSM8K 习惯给最终答案
    prompt = (
        question.strip()
        + "\n\nWrite a step by step solution to the problem with rigorous reasoning and explanations "
        + "(not just key points). Put the answer after the step-by-step solution, not before. "
        + "Please put your final answer in \\boxed{}."
    )
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new,
        do_sample=False,  # greedy：可复现
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()

def score_em(pred, gold):
    pn, gn = extract_final_number(pred), extract_final_number(str(gold))
    if pn is None or gn is None:
        return 0.0
    try:
        return float(float(pn) == float(gn))
    except Exception:
        return float(pn == gn)


In [ ]:
# 【步骤】跑基线；写出 gsm8k_baseline.json
scores = []
details = []
t0 = time.perf_counter()
for i, r in enumerate(rows):
    pred = generate(r["question"])
    s = score_em(pred, r["gold"])
    scores.append(s)
    if i < 8 or s == 0:
        details.append({
            "id": r["id"],
            "gold": r["gold"],
            "pred_num": extract_final_number(pred),
            "score": s,
            "pred_head": pred[:240],
        })
    if (i + 1) % 10 == 0:
        print(f"{i+1}/{len(rows)} running_em={sum(scores)/len(scores):.3f}")

em = sum(scores) / max(1, len(scores))
elapsed = time.perf_counter() - t0
report = {
    "notebook": "grok-009-gsm8k-eval-baseline",
    "phase": "deep_D_hard_eval",
    "model_id": MODEL_ID,
    "n": len(rows),
    "exact_match": em,
    "elapsed_seconds": elapsed,
    "decode": {"do_sample": False, "max_new_tokens": 256},
    "eval_path": str(EVAL / "gsm8k_em.jsonl"),
    "gate_for_010": "SFT 后 EM 相对本基线提升；并报告失败案例",
    "examples": details[:12],
}
(OUT / "gsm8k_baseline.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
(OUT / "grok009_results.json").write_text(json.dumps(report, indent=2, ensure_ascii=False))
print(json.dumps({k: report[k] for k in ["model_id","n","exact_match","elapsed_seconds","gate_for_010"]}, indent=2))
print("DONE grok-009")


## 学习检查清单
- GSM8K 的 gold 如何从 `####` 解析？  
- 为何评测必须 greedy、固定子集 seed？  
- 下一步 010：在 **同一 gsm8k_eval_v1** 上做 QLoRA SFT，只许涨这个 EM。  
